# 02. Keyword review

- 확인: 한일 전처리, 키워드 매칭, 중복 표현 억제
- 입력: 비공개 언어별 문서·언급 산출물
- 공개: 원문 문장이 담긴 실행 출력 제거


In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

research_root = os.environ.get("ESG_RESEARCH_ROOT")
if not research_root:
    raise RuntimeError("ESG_RESEARCH_ROOT에 비공개 ESG 연구자료 폴더를 지정하세요.")
comp_root = Path(research_root).expanduser() / "comp"
if not comp_root.is_dir():
    raise FileNotFoundError(comp_root)

def require_file(path):
    if not path.is_file():
        raise FileNotFoundError(f"필요한 입력 파일이 없습니다: {path}")
    return path

output_dir = comp_root / "analysis" / "output"
data = {}
for lang in ("kr", "jp"):
    docs_path = require_file(output_dir / f"docs_{lang}.csv")
    mentions_path = require_file(output_dir / f"mentions_{lang}_raw.csv")
    docs = pd.read_csv(docs_path, dtype={"doc_id": str})
    mentions = pd.read_csv(mentions_path, dtype={"doc_id": str})
    if not {"doc_id", "cleaned_text_len", "is_main"}.issubset(docs.columns):
        raise ValueError(f"{lang} 문서 산출물의 열 구성이 예상과 다릅니다.")
    if not {"doc_id", "alias", "canonical", "category", "sentence_text", "is_selected_longest"}.issubset(mentions.columns):
        raise ValueError(f"{lang} 매칭 산출물의 열 구성이 예상과 다릅니다.")
    data[lang] = {"docs": docs, "mentions": mentions}


## 전처리·매칭 개요

- 정제 후 문서 길이와 매칭 수 확인
- `selected`: 겹치는 표현 중 최장 표현만 채택


In [ ]:
overview = []
for lang, item in data.items():
    docs, mentions = item["docs"], item["mentions"]
    selected = mentions["is_selected_longest"] == True
    overview.append({"언어": lang, "문서": len(docs), "main 문서": int(docs["is_main"].sum()),
                     "정제 후 길이 중앙값": docs["cleaned_text_len"].median(),
                     "전체 매칭": len(mentions), "선택된 매칭": int(selected.sum())})
display(pd.DataFrame(overview))
display(pd.concat({lang: item["mentions"].loc[item["mentions"]["is_selected_longest"] == True, "category"].value_counts()
                   for lang, item in data.items()}, names=["언어", "범주"]).rename("언급 수").to_frame())


## 매칭 문맥

- 언어·범주·표현을 지정해 원문 문장 확인
- 기본값에서는 원문을 표시하지 않음


In [ ]:
review_lang = "kr"       # "kr" 또는 "jp"
review_category = None  # 예: "E", "G"; None이면 범주 제한 없음
review_alias = None     # 예: 특정 키워드; None이면 문장 출력 안 함

if review_lang not in data:
    raise ValueError("review_lang는 'kr' 또는 'jp'여야 합니다.")
matches = data[review_lang]["mentions"]
matches = matches[matches["is_selected_longest"] == True]
if review_category is not None:
    matches = matches[matches["category"] == review_category]
if review_alias is not None:
    matches = matches[matches["alias"] == review_alias]
    display(matches[["alias", "canonical", "category", "sentence_text"]].head(12))
else:
    display(matches.groupby(["category", "canonical"], dropna=False).size()
            .sort_values(ascending=False).head(20).rename("언급 수").to_frame())


## 확인할 점

- 일반 용례의 ESG 오탐 여부
- 한일 사전의 범주·표현 차이
